1. Track eksperimente: foldovi, HR@k, blend score, feature set.
2. Sačuvaj finalni model + metadata + feature list.
3. Imaš demo/proof za klijenta: vidi se historija eksperimenata i zašto je izabran model.
MLflow ima official LightGBM autologging i ručno logovanje metrika/modela, tako da ti ovo fino paše na postojeći notebook. Official docs: MLflow LightGBM autologging, mlflow.lightgbm API.

In [1]:
pip install mlflow


  Using cached mlflow-3.12.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.12.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.13.5-cp310-cp310-win_amd64.whl.metadata (8.4 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.7-cp38-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.14.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached gitpython-3.1.50-py3-none-any.whl.metadata (14 

In [2]:
mlflow ui --host 127.0.0.1 --port 5000

SyntaxError: invalid syntax. Perhaps you forgot a comma? (562307984.py, line 1)

In [ ]:
http://127.0.0.1:5000

Onda otvori:

http://127.0.0.1:5000

In [ ]:
import mlflow
import mlflow.lightgbm
import json

In [ ]:
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("next-best-policy-recommender")

In [ ]:
with mlflow.start_run(run_name=f"fold_{fold+1}", nested=False):
    mlflow.log_params({
        "fold": fold + 1,
        "n_splits": n_splits,
        "objective": "lambdarank",
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 20,
        "min_child_samples": 15,
        "reg_alpha": 0.1,
        "reg_lambda": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "blend_model_weight": 0.3,
        "blend_markov_weight": 0.7,
        "candidate_count": len(all_types),
        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "train_dataset_rows": len(train_dataset),
        "val_dataset_rows": len(val_dataset),
    })

    mlflow.set_tags({
        "project": "client-policy-prediction",
        "model_type": "LightGBM Ranker",
        "ranking": "model_markov_blend",
        "fold_type": "temporal",
        "ignore_fold_1_for_final_selection": "true",
    })

    model.fit(
        X_train, y_train,
        group=group_train,
        eval_set=[(X_val, y_val)],
        eval_group=[group_val],
        eval_at=[1, 3, 5],
        callbacks=[log_evaluation(20), early_stopping(stopping_rounds=50, verbose=False)]
    )

    preds_val = evaluate(model, X_val, y_val, group_val,
                         dataset=val_dataset, label=f'Fold {fold+1} VAL')

    model_score_norm = normalize_by_group(preds_val, group_val)
    markov_score_norm = normalize_by_group(val_dataset['markov_prob'].values, group_val)
    blended_score = 0.3 * model_score_norm + 0.7 * markov_score_norm

    mlflow.log_metrics({
        "model_hr1": hit_rate_at_k(preds_val, y_val.values, group_val, 1),
        "model_hr3": hit_rate_at_k(preds_val, y_val.values, group_val, 3),
        "model_hr5": hit_rate_at_k(preds_val, y_val.values, group_val, 5),
        "model_mrr5": mrr_at_k(preds_val, y_val.values, group_val, 5),
        "blend_hr1": hit_rate_at_k(blended_score, y_val.values, group_val, 1),
        "blend_hr3": hit_rate_at_k(blended_score, y_val.values, group_val, 3),
        "blend_hr5": hit_rate_at_k(blended_score, y_val.values, group_val, 5),
        "blend_mrr5": mrr_at_k(blended_score, y_val.values, group_val, 5),
        "val_renewal_share": future_pairs_val['target_is_renewal'].mean(),
    })

    feature_payload = {
        "features": avail_train,
        "categorical_features": [c for c in CATEGORICAL_FEATURES if c in avail_train],
        "dropped_features": ["target_is_renewal", "client_cluster"],
        "score_formula": "0.3 * normalized_model_score + 0.7 * normalized_markov_prob",
    }

    with open("feature_payload.json", "w") as f:
        json.dump(feature_payload, f, indent=2)

    mlflow.log_artifact("feature_payload.json")

    mlflow.lightgbm.log_model(
        model,
        name="lightgbm_ranker",
        input_example=X_val.head(5)
    )

In [ ]:
real_results_df = results_df[results_df['fold'] != 1]

with mlflow.start_run(run_name="cv_summary"):
    mlflow.log_metrics({
        "real_hr1_mean": real_results_df['hr1'].mean(),
        "real_hr1_std": real_results_df['hr1'].std(),
        "real_hr3_mean": real_results_df['hr3'].mean(),
        "real_hr3_std": real_results_df['hr3'].std(),
        "real_hr5_mean": real_results_df['hr5'].mean(),
        "real_hr5_std": real_results_df['hr5'].std(),
    })

    mlflow.log_param("selected_fold", best_fold['fold'])
    mlflow.log_param("selection_metric", "best HR@3 excluding fold 1")
    mlflow.log_param("final_scoring", "0.3 model + 0.7 markov")

 Šta pokazati klijentu

U MLflow UI možeš pokazati:

- fold_2, fold_3, fold_4 rezultate
- blend_hr1, blend_hr3, blend_hr5
- feature importance/model artifact
- parametre modela
- da Fold 1 nije korišten za finalni izbor

Za prodaju bih u MLflow-u imao eksperimente nazvane čisto:

next-best-policy-recommender
i metrike:

blend_hr1
blend_hr3
blend_hr5
renewal_hr1
non_renewal_hr3
non_renewal_hr5

Za ozbiljnu produkcijsku prodaju bih dodao još dashboard/API demo i business metrike tipa uplift u konverziji, ali kao tehnički POC ovo sada ima smisla.

Kako bi ti ovo uradio?
6:23 PM

Ja bih ovo uradio kao prodajni pilot u 3 sloja: model tracking, API, dashboard + business mjerenje. Ne bih odmah pravio ogromnu produkciju. Napravio bih nešto što klijent može vidjeti, kliknuti i razumjeti vrijednost.

1. MLflow kao dokaz ozbiljnosti
Prvo bih zaključao eksperiment u MLflow:

Experiment: next-best-policy-recommender
Runs: fold_2, fold_3, fold_4, final_model
Metrics: HR@1, HR@3, HR@5, MRR@5, renewal HR, non-renewal HR
Artifacts: model, feature list, scoring formula, feature importance
Ovo pokazuje da model nije “jedan notebook”, nego reproducibilan eksperiment.

2. API demo
Napravio bih mali FastAPI servis sa endpointom:

POST /recommend
Input:

{
  "klijent_id": 12345,
  "current_ponuda_id": 999,
  "current_type": 6
}
Output:

{
  "klijent_id": 12345,
  "recommendations": [
    {
      "candidate_type": 6,
      "score": 0.91,
      "reason": "Klijent često obnavlja ovaj tip polise"
    },
    {
      "candidate_type": 1,
      "score": 0.63,
      "reason": "Slični klijenti često prelaze na ovaj tip"
    },
    {
      "candidate_type": 7,
      "score": 0.48,
      "reason": "Visoka Markov vjerovatnoća prelaza"
    }
  ]
}
U pozadini API koristi:

final_score = 0.3 * model_score_norm + 0.7 * markov_prob_norm
To je najbolji blend koji si našao.

3. Dashboard demo
Za dashboard bih koristio Streamlit za brz pilot. Stranice:

1. Client Lookup
2. Top 5 Recommendations
3. Explanation
4. Portfolio / history
5. Model performance
Na ekranu za agenta bih prikazao:

Klijent: ID, starost, regija, broj prethodnih polisa
Trenutna polisa: sif_vrsta
Top 5 preporuka
Score
Da li je renewal kandidat
Markov probability
Objašnjenje: "zašto ovo preporučujemo"
Ne bih ga pravio kao ML dashboard za data scientiste, nego kao agent assist alat.

4. Business metrike
Ovo je najvažniji dio za produkcijsku prodaju. Tehničke metrike su dobre, ali direktor želi čuti novac.

Mjerio bih:

Conversion rate:
Koliko preporučenih ponuda se pretvori u prodaju.

Top-3 assisted conversion:
Koliko prodaja je bilo unutar top 3 preporuke.

Agent adoption:
Koliko često agent klikne/preuzme preporuku.

Revenue uplift:
Da li grupa sa preporukama ima veću premiju/prodaju od kontrolne grupe.

Time saved:
Koliko se smanjuje vrijeme pripreme ponude.

Cross-sell uplift:
Da li raste prodaja non-renewal/cross-sell proizvoda.
5. Pilot/A-B test
Najbolji prodajni eksperiment:

Grupa A: agenti rade kao i do sada
Grupa B: agenti vide top 3 preporuke
Trajanje: 4-8 sedmica
Mjeri se: conversion, premium, cross-sell, renewal retention
Ako grupa B ima npr. 5-10% bolju konverziju ili više premiuma, to je prodajni argument.

6. Kako bih ga upakovao
Nazvao bih ga:

Next Best Policy Assistant
Poruka za klijenta:

Sistem analizira historiju klijenta, ponašanje sličnih klijenata i obrasce prelaza između vrsta polisa, te agentu daje top 3-5 preporuka za sljedeću najvjerovatniju ponudu.
Prodajna tvrdnja, fer i jaka:

Na realnijoj vremenskoj validaciji, sistem stavlja stvarnu narednu vrstu polise u top 3 preporuke u oko 73-75% slučajeva, a u top 5 u oko 78-80% slučajeva uz blend ranking.
Moj redoslijed rada bi bio:

1. Srediti MLflow tracking.
2. Sačuvati final model + scoring config.
3. Napraviti FastAPI /recommend.
4. Napraviti Streamlit dashboard za agenta.
5. Dodati logging svake preporuke.
6. Mjeriti konverziju kroz pilot.